### `phi_scalar.ipynb` 
*Created: September 22, 2026*

This notebook implements a function for evaluating the phi functions with complex scalar arguments: 
\begin{align*}
   \varphi_m(z) &= \sum_{k=0}^{\infty} \frac{z^k}{(k+m)!} = z^{-m}\left(e^z - 1 - z - \frac{z^2}{2!} - \cdots - \frac{z^{m-1}}{(m-1)!} \right)   \qquad \textrm{when } z \neq 0
\end{align*}

We define $\varphi_m(0) := 0$. The implementation in this notebook evaluates $\varphi_m(z)$ using the Taylor series when $|z|$ is small. If $|z|$ is not small, then $\varphi_m(z)$ is evaluated using the recursive formula

$$\phi_{m+1}(z) = \frac{\phi_m(z) - \frac{1}{m!}}{z}$$

In [1]:
using LinearAlgebra, Random, Statistics, NBInclude

In [2]:
function phis(z::Number, k::Int64; ϵ::Float64 = 1e-3)
    """
    Given a complex number z and a nonnegative integer k, 
    compute the vector [φ_0(z), φ_1(z),...,φ_k(z)]. 
    
    Note that φ_k(z) is a scalar for each k.  

    PARAMETERS
    ----------
    z :: a complex number 
    k :: a nonnegative integer
    ϵ :: if |k| ≤ ϵ, use recursive formula to compute φ_k(z) for k ≥ 1.

    RETURNS
    -------
    φ = [φ_0(z),φ_1(z),…,φ_k(z)]    (vector of length k + 1) 
    """
    
    #Exception handling 
    k ≥ 0 || throw(ArgumentError("k must be nonnegative. Passed k = $k."))
    k ≤ 6 || throw(ArgumentError("To be safe, it is required that k ≤ 6."))
    
    #Initialize vector to store the ϕ_j's 
    φ = Vector{typeof(exp(z))}(undef, k+1)

    #Compute ϕ_0(z) = e^z using built-in exp() function 
    φ[1] = exp(z)  

    #If k = 0, we're done
    if k == 0
        return φ    
    end 

    ################################################################
    #If |z| is small, use Taylor series to compute ϕ_j(z) for j ≥ 1
    ################################################################
    if abs(z) ≤ ϵ   
        M = 5  #Use Taylor expansion with M + 1 terms 
        powers = [z^j for j=0:M]
        coeffs = 1 ./ [factorial(j) for j=0:k+M]
        
        for j = 1:k
            φ[j+1] = dot(coeffs[j+1:j+M+1], powers)    
        end 
    
    else #If |z| is not small, use recursive formula instead
        for j= 1:k
            φ[j+1] = (φ[j] - 1/factorial(j-1)) / z    
        end 
    end

    #### NOTE! ###
    #Because of the indexing, the above recursive formula looks wrong, but it's not! 
    #We have ϕ[j+1] = ϕ_j(z) and ϕ[j] = ϕ_{j-1}(z) for j = 1,...,k, and hence 
    #ϕ_j(z) = (ϕ_{j-1}(z) - 1/(j-1)!) / z , which is correct! 
    
    return φ
end 

phis (generic function with 1 method)

In [18]:
function phi(z::Float64; precision::Int = 64)
    """
    Compute ϕ₀(z) = (e^z - 1) / z in high precision.

    z :: value at which to evaluate ϕ₀
    precision :: number of bits in the BigFloat significand 

    Approximate Decimal Precision:
        64 bits ≈ 19 decimal digits
        256 bits ≈ 77 decimal digits 
    
       log10(2.0^64) = 19.2659...
       log10(2.0^256) ≈ 77.0

    Note: Float64 uses 64 bits of storage, but has 53 bits of significand precision, 
          corresponding to roughly 16 decimal digits.
    """

    return setprecision(BigFloat, precision) do 

        #Convert z to a BigFloat of the desired precision
        z_big = BigFloat(z)   

        if iszero(z_big)
            return one(z_big)
        end 
    
        return expm1(z_big) / z_big
    end 
end 


function precision_comparison(zvals::Vector{Float64}, precision)
  
    #Compare Float64 evaluation against a BigFloat reference, using the specified precision. 

    return setprecision(BigFloat, precision) do 

        #Compute phis in high precision 
        phis_high_precision = [phi(z; precision = precision) for z in zvals]

        #These operations use Float64 arithmetic. **See "Important Note" below!**
        #phis_reg_precision = [iszero(z) ? one(z) : expm1(z) / z for z in zvals]

        phis_reg_precision = [iszero(z) ? one(z) : (exp(z) - 1) / z for z in zvals]

        #=Important Note!
        - The types of the operands determine the arithmetic used. 
        - The setprecision block only controls the precision of operations that already involve BigFloats!
         Since the elements of `zvals` are Float64s (not BigFloat's), the computation within the 
         array comprehension is regular old Float64 arithmetic.  
        =#

        #Now compute relative errors, converting `phis_reg_precision` to high precision before the subtraction,
        #so that the relative errors are computed in high precision. 
        rel_errors = abs.((phis_high_precision .- BigFloat.(phis_reg_precision)) ./ phis_high_precision)

        return rel_errors
    end 
end 

precision_comparison (generic function with 4 methods)

In [17]:
zvals = [1e2, 1e1, 1e0, 1e-1, 1e-2, 1e-4, 1e-8, 1e-16, 1e-32, 1e-64]
rel_errors = precision_comparison(zvals, 256)

10-element Vector{BigFloat}:
 1.059502631005634954109337919998999926168283202969243086597260217176798039977621e-16
 1.038574254517707648259426507360515300256179778487757075830607541910739025922681e-16
 4.509150621792367191626215341510041829575758799952047008041388801969312971212751e-17
 1.411458724018264208132127738270683944656821394994443867476503347663349492619106e-17
 3.842201081495313306462005252502962061425265266242720911487929556081146418740576e-17
 9.82500127730204050093323367239925717936097187386820639801214194554982384496e-17
 1.74990582382521506401842493623985503716378183819657063003053209306578502426477e-16
 4.999999999999999812155600286839687959574454031799732688535199538556755602143797e-17
 5.000000000000000279836549881209501338927879689156330376410499301218030276763308e-33
 5.000000000000164284816608563896943402270082848700437677258631187855978673084319e-65

In [ ]:
f

In [ ]:
function foo(x)
    iszero(x) ? return 0.0 : return 1.0
end

In [ ]:
phis(1.0, 5)

In [ ]:
collect(range(start = 0.1, step = 0.5, length = 4))

In [ ]:
sum([1,3,4,5])

In [ ]:
#Scalar phi's 
phi0(z) = exp(z)
phi1(z) = expm1(z) / z 
phi2(z) = (expm1(z) - z) / z^2 
phi3(z) = (expm1(z) - z - z^2/2) / z^3
phi4(z) = (expm1(z) - z - z^2/2 - z^3/6) / z^4

